# 生产推理与模型部署：从不可变制品到可回滚的在线服务

> **本章定位**：将通过 `50_model_evaluation.ipynb` 部署准入评估的模型产物组织为具有稳定推理契约、流式调用、弹性伸缩、可观测性和回滚能力的受控服务。

> **章节边界**：本章以 [50_model_evaluation.ipynb](50_model_evaluation.ipynb) 形成的 Evaluation Manifest 为输入，建立“模型制品 → 推理请求 → 调度与 Worker → 流式响应”的部署主链路。第三方开放权重模型还要消费 [E10_open_model.ipynb](E10_open_model.ipynb) 形成的 Model Audit Manifest，并与评测阶段绑定的摘要一致。KV Cache、FlashAttention、编译、连续批处理和推理并行见 [A50_inference_optimization.ipynb](A50_inference_optimization.ipynb)；完整威胁建模、纵深控制和最终放量门禁见 [70_model_safety.ipynb](70_model_safety.ipynb)。

本章代码刻画制品、请求、状态和容量契约。完整在线服务还需要模型服务器、网关、编排平台与可观测性平台共同实现。通过本章验收表示服务具备受控部署能力，不等同于获准全面生产开放。


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 训练与推理系统：推理服务 |
| 本章定位 | 将模型产物接入稳定 API、受控发布、扩缩容、观测和故障恢复链路。 |
| 先修知识 | 完成 `50`；理解模型 ID、制品哈希、生成配置、评测基线与部署准入门禁。部署第三方开放权重模型时，先完成 `E10` 的模型仓库审计。 |
| 预计时间 | 2～3 小时 |
| 运行资源 | 架构与控制面实验可在 CPU 运行；真实容量取决于目标硬件压测。 |
| 输入 | 不可变模型制品、生成配置、请求契约和 Evaluation Manifest；第三方开放权重模型另需 Model Audit Manifest。 |
| 交付物 | 版本化服务契约、组件边界、受控发布策略、SLO、最低安全门禁与故障矩阵。 |

### 1.1．学习目标

完成本章后，读者能够定义模型制品与推理 API，解释 Gateway、Router、Scheduler 和 Worker 的职责，建立容量与可观测证据，并设计可灰度、可回滚的发布流程。


### 1.2．系统依赖与交付路径

```mermaid
flowchart LR
  A["训练与微调产物"] --> B["固化制品契约"]
  O["第三方开放权重仓库"] --> M["Model Audit Manifest"]
  M --> B
  B --> C["注册不可变修订"]
  C --> D["加载与预热"]
  D --> E["开放就绪流量"]
  E --> F["流式推理与取消"]
  F --> G["指标、日志与追踪"]
  G --> H["扩缩容与故障治理"]
  H --> I["灰度、滚动或回滚"]
  I --> C
```

原理单元使用 Python 数据对象、Pydantic Schema 和容量公式表达系统契约；生产迁移对应模型注册表、标准 API、模型服务引擎、编排平台与 OpenTelemetry。四类证据分别回答制品可重现性、请求状态一致性、容量是否满足体验约束，以及候选修订能否安全晋级。


## 2．直觉与输入输出契约

### 2.1．部署范围与系统契约

| 领域 | 解决的问题 | 本章处理方式 |
|---|---|---|
| 模型部署 | 制品、接口、生命周期、容量、发布和运维 | 本章主线 |
| 推理优化 | 降低单次执行与调度的计算、访存和空槽开销 | 复用其引擎能力与指标；原理见 [A50_inference_optimization.ipynb](A50_inference_optimization.ipynb) |
| 模型压缩 | 量化、剪枝、低秩分解与蒸馏对模型产物的影响 | 作为新的不可变制品重新验收和发布 |
| 模型安全 | 输入、输出、工具、租户、数据与供应链的控制 | 最低安全门禁在本章接入；系统级保障见 [70_model_safety.ipynb](70_model_safety.ipynb) |

生产部署的最小单位由以下对象共同构成：**不可变制品 + 明确 API + 可治理服务拓扑 + 生命周期状态机 + 发布与回滚策略 + 可观测证据**。


<!-- theory-math-contract:v1 -->
### 2.2．核心机制的语言与数学表达

在线生成延迟由排队、Prefill、首 Token 返回和逐 Token Decode 共同构成。对输出 $N_{\mathrm{out}}$ 个 Token 的请求，可用以下关系建立容量基线：

$$
T_{\mathrm{e2e}}=T_{\mathrm{queue}}+T_{\mathrm{TTFT}}+(N_{\mathrm{out}}-1)T_{\mathrm{TPOT}},
\qquad \rho=\frac{\lambda}{c\mu}<1
$$

其中，$\lambda$ 是到达率，$\mu$ 是单 Worker 服务率，$c$ 是并行 Worker 数，$\rho$ 是利用率。`request_deadline`、Scheduler 队列和 Worker 指标分别承载这些变量。$\rho<1$ 只是稳态必要条件，不保证尾延迟达标；批处理、上下文长度、故障冗余和流量突发都必须进入容量压测。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．不可变模型制品

模型目录只是文件集合，制品契约才说明这些文件如何共同解释。一个可部署制品至少固定：

- 权重文件及摘要，优先使用不执行任意代码的安全序列化格式；
- 模型结构配置、Tokenizer、特殊 Token、聊天模板和生成默认值；
- 第三方开放权重模型的 Model Audit Manifest 摘要，且其仓库文件摘要和制品组合与 Evaluation Manifest 一致；
- Adapter、量化参数或其他派生产物的基座修订与组合顺序；
- 精度、最大上下文、运行时镜像摘要和最低资源轮廓；
- 训练代码修订、数据谱系、许可证、质量报告与安全报告；
- 每个文件的内容摘要，以及整个清单的签名。

环境名、流量权重、配额和动态策略不写进模型制品；它们属于部署配置，并单独修订。这样，同一制品可以在不同环境晋级，而不会被重新打包。


#### 3.1.1．制品内部关系

![架构图：不可变制品清单约束加载器与已预热模型服务器](assets/figures/60_inference_deployment/artifact-boundary.svg)

[TikZ 源文件](assets/figures/60_inference_deployment/artifact-boundary.tex)

清单是唯一入口。模型服务器不应根据目录中未经清单声明的文件猜测加载方式，也不应在启动时静默下载浮动修订。


In [ ]:
# 用 Pydantic 把制品依赖固化为机器可读契约；frozen 保证对象创建后不可改写。
from typing import Annotated, Literal

from pydantic import BaseModel, ConfigDict, Field


class MyArtifactFile(BaseModel):
    """描述模型制品中单个文件的角色、摘要与字节大小。"""
    model_config = ConfigDict(frozen=True, extra="forbid")

    path: str
    role: Literal["weights", "config", "tokenizer", "template", "metadata", "report"]
    digest: str
    size_bytes: int = Field(gt=0)


class MyRuntimeContract(BaseModel):
    """固化部署镜像、精度、上下文上限与加速器容量契约。"""
    model_config = ConfigDict(frozen=True, extra="forbid")

    image_digest: str
    precision: Literal["fp32", "fp16", "bf16", "int8", "int4"]
    max_context_tokens: int = Field(gt=0)
    accelerator_memory_gib: int = Field(gt=0)


class MyArtifactManifest(BaseModel):
    """聚合模型 ID、文件清单和兼容运行时的不可变 Manifest。"""
    model_config = ConfigDict(frozen=True, extra="forbid")

    model_id: str
    architecture: str
    model_audit_manifest_sha256: str | None = Field(default=None, pattern=r"^[0-9a-f]{64}$")
    files: tuple[MyArtifactFile, ...]
    runtime: MyRuntimeContract


# 示例使用逐文件内容摘要标识已加载制品。
artifact_manifest = MyArtifactManifest(
    model_id="acme/chat-model",
    architecture="causal-lm",
    files=(
        MyArtifactFile(
            path="model.safetensors",
            role="weights",
            digest="sha256:91bf23e8",
            # 8.4×10^9 Byte 是制品清单夹具；生产值必须由实际权重文件生成。
            size_bytes=8_400_000_000,
        ),
        MyArtifactFile(
            path="tokenizer.json",
            role="tokenizer",
            digest="sha256:b620f135",
            # 2.1×10^6 Byte 是 Tokenizer 夹具大小；文件变更后必须重新计算。
            size_bytes=2_100_000,
        ),
    ),
    runtime=MyRuntimeContract(
        image_digest="sha256:13cf0ab9",
        precision="bf16",
        # 32,768 Token 是服务总上下文上限；不得超过模型与后端共同支持值，调高会增加 KV Cache。
        max_context_tokens=32_768,
        # 80 GiB 是容量规划硬件画像；更换加速器后须重算权重、KV Cache 与运行时余量。
        accelerator_memory_gib=80,
    ),
)

artifact_manifest.model_dump(mode="json")


#### 3.1.2．注册、晋级与加载

制品注册表保存内容摘要、签名和报告；部署控制面只引用摘要。推荐路径是：

1. 构建阶段生成清单并计算摘要；
2. 质量、安全和许可证门禁通过后注册；
3. 部署环境按摘要拉取，先校验清单，再加载权重；
4. 使用固定形状集合完成预热，建立真实的启动耗时和显存基线；
5. 预热完成且依赖可用后，才把实例标记为就绪。

已有修订保持不可变。即使只改聊天模板、Tokenizer 或量化参数，也应产生新的制品摘要并重新走兼容性与质量门禁。


### 3.2．稳定的推理 API

生产 API 首先固定语义，再选择 HTTP、gRPC 或消息协议。本章选择：

- 外部同步请求使用 HTTP；
- 单向 Token 流使用 Server-Sent Events（SSE），便于经过常见网关和代理；
- 内部组件使用支持期限传播与取消的 RPC；
- 只有确实需要双向交互时才采用 WebSocket。

API 不暴露引擎私有参数。外部契约保持稳定，Router 再把通用字段翻译成具体模型服务器的请求。


#### 3.2.1．端点与语义

| 端点 | 语义 | 关键约束 |
|---|---|---|
| `POST /generate` | 完整响应 | 客户端期限小于网关最大期限；返回实际制品修订 |
| `POST /generate/stream` | SSE 流式响应 | 事件有单调序号；`done` 或 `error` 是终止事件 |
| `DELETE /requests/{request_id}` | 取消请求 | 幂等；取消信号必须传播到队列和 Worker |
| `GET /health/startup` | 启动完成 | 权重加载与预热尚未完成时不通过 |
| `GET /health/live` | 进程仍可推进 | 只判断不可恢复的卡死，不把瞬时过载当成死亡 |
| `GET /health/ready` | 可以接收新流量 | Draining、过载或关键依赖失效时退出就绪 |
| `GET /metrics` | 暴露聚合指标 | 不包含提示词、用户标识等高基数或敏感内容 |

`request_id` 由服务端生成并贯穿日志、追踪、取消和计费；客户端可以另传幂等键。响应通过 `request_id` 关联服务端保存的制品文件摘要和运行 Trace。


In [ ]:
# 外部请求只表达稳定语义；Router 负责把它翻译成具体推理引擎参数。
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field


class MyChatMessage(BaseModel):
    """表示生成请求中的单条受限角色消息。"""
    model_config = ConfigDict(extra="forbid")

    role: Literal["system", "user", "assistant"]
    content: str


class MyGreedyParameters(BaseModel):
    """声明不使用随机采样的 Greedy 解码参数。"""
    model_config = ConfigDict(extra="forbid")

    mode: Literal["greedy"] = "greedy"


class MySamplingParameters(BaseModel):
    """声明带 Temperature、Top-p 和可选 Seed 的采样策略。"""
    model_config = ConfigDict(extra="forbid")

    # 判别字段将随机采样与 Greedy 分离；切换模式属于生成策略版本变更。
    mode: Literal["sample"] = "sample"
    # 0.8 控制分布平滑度；提高通常增加多样性与错误风险，须与 Top-p 联合评测。
    temperature: float = Field(default=0.8, gt=0.0)
    # 0.95 保留累计概率 95% 的候选集合；降低会收窄候选，任务或模型变化后重校准。
    top_p: float = Field(default=0.95, gt=0.0, le=1.0)
    # None 表示不承诺采样重放；显式 Seed 仅定位随机序列，不作为质量参数。
    seed: int | None = None


MyDecodingParameters = Annotated[
    MyGreedyParameters | MySamplingParameters,
    Field(discriminator="mode"),
]


class MyGenerateRequest(BaseModel):
    """定义网关接收的版本化文本生成请求契约。"""
    model_config = ConfigDict(extra="forbid")

    model: str
    messages: list[MyChatMessage]
    # 256 Token 限制单请求输出；调高会增加 KV、费用与尾延迟，调低需复核截断率。
    max_new_tokens: int = Field(default=256, gt=0)
    stop: list[str] = Field(default_factory=list)
    stream: bool = True
    # 30,000 ms 是端到端期限；须按 SLO 分配给排队、Prefill、Decode 与网络。
    deadline_ms: int = Field(default=30_000, gt=0)
    idempotency_key: str | None = None
    decoding: MyDecodingParameters = Field(default_factory=MySamplingParameters)


class MyTokenUsage(BaseModel):
    """记录一次生成的输入与输出 Token 用量。"""
    input_tokens: int
    output_tokens: int


class MyGenerateResponse(BaseModel):
    """定义非流式生成结果、结束原因和 Token 用量。"""
    request_id: str
    text: str
    finish_reason: Literal["stop", "length", "cancelled", "error"]
    usage: MyTokenUsage


class MyStreamEvent(BaseModel):
    """定义流式生成生命周期中的有序事件。"""
    request_id: str
    sequence: int
    event: Literal["created", "delta", "usage", "done", "error"]
    text_delta: str | None = None
    finish_reason: str | None = None


# 契约对象可直接生成 JSON Schema，供网关、SDK 和服务端共享。
request_schema = MyGenerateRequest.model_json_schema()
request_schema


#### 3.2.2．流式返回与取消时序

```mermaid
sequenceDiagram
  actor C as 客户端
  participant G as Gateway
  participant R as Router
  participant S as Scheduler
  participant W as Worker
  C->>G: POST /generate/stream
  G->>R: 身份、配额、期限与追踪上下文
  R->>S: 解析当前活动部署后入队
  S->>W: 调度请求
  W-->>S: Token 增量
  S-->>R: Token 增量
  R-->>G: SSE delta
  G-->>C: sequence 与文本增量
  C->>G: DELETE /requests/{request_id}
  G->>R: 传播取消
  R->>S: 从队列移除或标记中止
  S->>W: 停止后续解码
  W-->>G: 最终 usage 与 cancelled
  G-->>C: 终止事件
```

取消不是关闭客户端连接就结束。Gateway、Router、Scheduler 和 Worker 都要识别同一个 `request_id`，释放排队槽位、计算资源和计费状态。


#### 3.2.3．请求状态、期限与错误

请求状态建议固定为 `accepted → queued → running → completed | cancelled | failed | expired`。状态转换只向前，终止状态不可再次进入队列。

- **期限传播**：客户端期限转换为绝对截止时间，逐跳扣除已经消耗的时间；过期请求不再进入昂贵计算。
- **幂等边界**：尚未开始流式返回时，可以在受控条件下重试；已经输出 Token 后自动重试会生成重复或分叉内容。
- **错误契约**：稳定错误码区分无效请求、配额不足、队列过载、期限到期、修订不可用和内部故障；响应不暴露堆栈。
- **背压感知流**：客户端读取过慢时限制发送缓冲；超出预算后取消请求，避免单个连接占满内存。

`429` 表示租户配额或速率预算不足，`503` 表示当前服务容量不可用；两者应携带可解释的重试提示，但客户端仍需使用抖动退避。


### 3.3．Gateway、Router、Scheduler 与 Worker 的职责

![架构图：推理服务的数据面、制品面、控制面与遥测面](assets/figures/60_inference_deployment/serving-planes.svg)

[TikZ 源文件](assets/figures/60_inference_deployment/serving-planes.tex)

数据面处理请求，控制面管理期望状态。Router 使用经过校验且可原子替换的本地路由快照，可避免每个请求同步查询注册表或配置中心。


#### 3.3.1．组件职责与不变量

| 组件 | 核心职责 | 必须保持的不变量 |
|---|---|---|
| Gateway | TLS、鉴权、配额、请求大小、协议转换、审计入口 | 不决定具体 Worker；不记录原始敏感内容 |
| Router | 解析模型别名、固定制品修订、选择区域与故障域 | 一个请求生命周期内不切换修订 |
| Scheduler | 有界排队、租户公平、优先级、批调度、取消 | 不接收超过容量预算且无法按期限完成的请求 |
| Worker | 加载一个明确制品、执行推理、报告 Token 与资源指标 | 未完成预热不进入 Ready；Draining 后不接新请求 |
| 部署控制面 | 期望副本、扩缩容、发布、回滚和配置分发 | 变更可审计，修订可重现 |

Scheduler 可以调用推理优化章节选定的连续批处理、分页缓存或前后缀复用能力，但这些算法不是部署控制面的职责。部署侧关心的是兼容性、容量信号、取消语义和故障隔离。


#### 3.3.2．Worker 生命周期与三类健康信号

```mermaid
flowchart LR
  C["Created"] -->|"拉取并校验制品"| L["Loading"]
  L -->|"权重加载完成"| W["Warming"]
  W -->|"预热与依赖就绪"| R["Ready"]
  R -->|"暂时过载或关键依赖异常"| N["NotReady"]
  N -->|"恢复"| R
  R -->|"发布、缩容或维护"| D["Draining"]
  N -->|"计划下线"| D
  D -->|"在途请求结束或宽限期到期"| T["Terminated"]
  L -->|"制品或资源故障"| F["Failed"]
  W -->|"预热失败"| F
  R -->|"不可恢复卡死"| F
```

| 信号 | 回答的问题 | 失败后的动作 |
|---|---|---|
| Startup | 权重是否加载并完成预热？ | 保持隔离；超过启动预算后重建实例 |
| Liveness | 进程是否还能向前推进？ | 重启实例；不能把排队变长直接视为死亡 |
| Readiness | 此刻是否应该接收新请求？ | 从服务端点移除，保留进程以恢复或排空 |

大型模型启动慢，Startup 必须覆盖下载、校验、加载和预热时间。把 Liveness 配得过于敏感，会在过载时反复重启并放大故障。


#### 3.3.3．优雅下线时序

```mermaid
sequenceDiagram
  participant C as 部署控制面
  participant R as Router
  participant S as Scheduler
  participant W as Worker
  C->>W: 标记 Draining
  W-->>R: Readiness 退出
  R->>S: 停止分配新请求
  S->>W: 仅保留已调度请求
  W-->>S: 持续返回在途 Token
  S-->>C: 在途数量归零
  C->>W: 结束进程
  Note over C,W: 超过宽限期后取消剩余请求并记录终止原因
```

正确顺序是：**先停止新流量，再排空队列与在途请求，最后终止进程**。宽限期至少覆盖服务允许的最大请求期限，并为指标冲刷、追踪导出和连接关闭留出余量。


## 4．证据验证

### 4.1．不可变修订的发布与回滚

发布对象由制品摘要、运行时镜像摘要、部署配置摘要和路由策略共同确定。任何一项改变都形成新的候选修订，但不会覆盖当前稳定修订。

```mermaid
flowchart LR
  A["构建并签名候选制品"] --> B["静态门禁与回归评测"]
  B --> C["注册不可变修订"]
  C --> D["影子流量：不返回用户"]
  D --> E["小比例灰度"]
  E --> F["逐步扩大流量"]
  F --> G["成为稳定修订"]
  D -->|"资源或质量异常"| R["停止候选修订"]
  E -->|"SLO 或质量回归"| R
  F -->|"SLO 或质量回归"| R
  R --> H["路由回稳定修订"]
  H --> I["保留证据并复盘"]
```

影子流量适合比较延迟、错误和资源，但会增加成本，并且复制真实请求前必须满足隐私与数据使用要求。


#### 4.1.1．发布策略的组合与适用条件

| 策略 | 适合场景 | 主要风险 |
|---|---|---|
| 滚动更新 | 同一模型修订的大规模副本替换 | 新旧副本并存，需要协议与状态兼容 |
| 流量灰度 | 模型、模板、Tokenizer 或运行时行为变化 | 样本偏差会掩盖质量回归 |
| 蓝绿切换 | 需要最快回滚且容量允许双份驻留 | 设备成本高，切换瞬间需控制连接排空 |
| 影子验证 | 在真实分布下比较资源与离线质量 | 不代表真实用户体验，需治理复制数据 |

常见组合是：先影子验证，再让候选修订接收少量真实流量，最后通过滚动方式替换剩余副本。门禁至少覆盖：

- 请求错误率、排队时间、TTFT 和 TPOT 的尾分位；
- 输出质量、安全违规率、拒答率与任务成功率；
- 峰值显存、每秒 Token、单位 Token 成本与启动时间；
- 取消成功率、超时率、健康状态抖动和故障恢复时间。


#### 4.1.2．回滚对象与一致性要求

回滚对象覆盖完整服务契约，包括：制品摘要、运行时镜像、聊天模板、生成默认值、路由权重和兼容的部署配置。

回滚路径应在每次发布前演练，并满足三个条件：

1. 稳定修订仍可拉取，且保留足够热容量；
2. Router 可以原子切换路由快照，新请求立即固定到稳定修订；
3. 候选修订进入 Draining，已流式返回的请求按原修订完成或明确终止，不在中途迁移。

数据格式、会话状态或工具协议存在不兼容变化时，应先设计双读、双写或兼容窗口；否则模型回滚成功，外围系统仍可能失败。


### 4.2．Token 负载与队列容量规划

设备利用率是结果，不是完整需求信号。LLM 请求的输入长度、输出长度和生成阶段不同，相同请求数可能对应完全不同的计算量。先建立三个层次的容量模型：

1. **离线基准**：在目标制品、引擎、硬件和长度分布下测量每个 Worker 的可持续 Token 吞吐；
2. **在线负载**：按模型修订与租户聚合到达率、输入 Token 和输出 Token；
3. **体验约束**：用排队时间、TTFT、TPOT 和端到端尾延迟校准目标利用率。

平均值用于粗估容量，尾分位用于保护体验；二者不能互相替代。


#### 4.2.1．扩缩容控制环

```mermaid
flowchart LR
  M["采集：队列、Token、TTFT、TPOT、Ready 副本"] --> A["聚合：窗口、分位数与修订维度"]
  A --> D["决策：期望副本与扩缩速率"]
  D --> P["配置新副本"]
  P --> L["拉取、加载与预热"]
  L --> R["进入 Ready 并接收流量"]
  R --> M
  D --> Q["缩容候选进入 Draining"]
  Q --> M
```

模型副本从创建到 Ready 可能需要数分钟，扩缩容决策必须包含启动提前量。快速扩容、保守缩容、缩容稳定窗口和最小热容量可以降低抖动。


#### 4.2.2．基础容量模型

设请求到达率为 $\lambda$，平均输入与输出长度分别为 $E[T_{in}]$、$E[T_{out}]$，则平均 Token 需求近似为：

$$
D_{token} = \lambda \left(E[T_{in}] + E[T_{out}]\right)
$$

若基准测得单个 Worker 的可持续吞吐为 $Q_{worker}$，目标利用率为 $u$，粗略副本数为：

$$
R = \left\lceil \frac{D_{token}}{Q_{worker}u} \right\rceil
$$

Little 定律把平均在途请求数 $L$、到达率 $\lambda$ 和平均停留时间 $W$ 联系起来：

$$
L = \lambda W
$$

这些公式只用于初始容量预算。生产校准还要区分 Prefill 与 Decode、长度分桶、批调度效率、尾延迟和故障冗余。相关执行原理见 [A50_inference_optimization.ipynb](A50_inference_optimization.ipynb)。


In [ ]:
# 用实测 Worker 吞吐给出第一版副本预算；目标利用率为尾延迟保留余量。
import math


def my_plan_replicas(
    arrival_requests_per_second: float,
    mean_input_tokens: float,
    mean_output_tokens: float,
    measured_worker_tokens_per_second: float,
    target_utilization: float,
) -> int:
    """根据请求 Token 需求、实测吞吐和目标利用率规划副本数。"""
    token_demand_per_second = arrival_requests_per_second * (
        mean_input_tokens + mean_output_tokens
    )
    usable_worker_capacity = measured_worker_tokens_per_second * target_utilization
    return math.ceil(token_demand_per_second / usable_worker_capacity)


# 吞吐参数必须来自目标制品、引擎、硬件和请求分布的基准结果。
capacity_profile = {
    "arrival_requests_per_second": 18.0,  # 夹具到达率，单位 requests/s；生产按流量分位数替换。
    "mean_input_tokens": 620.0,           # 单请求输入 Token；长度分布变化后重新测量。
    "mean_output_tokens": 180.0,          # 单请求输出 Token；解码策略变化后重新测量。
    "measured_worker_tokens_per_second": 3_800.0,  # 目标制品与硬件的实测吞吐，单位 tokens/s。
    "target_utilization": 0.70,  # 为突发、抖动和故障预留 30%；提高会加剧排队与尾延迟风险。
}
replicas = my_plan_replicas(**capacity_profile)
token_demand_per_second = capacity_profile["arrival_requests_per_second"] * (
    capacity_profile["mean_input_tokens"]
    + capacity_profile["mean_output_tokens"]
)
usable_worker_capacity = (
    capacity_profile["measured_worker_tokens_per_second"]
    * capacity_profile["target_utilization"]
)
capacity_plan = {
    "profile": capacity_profile,
    "token_demand_per_second": token_demand_per_second,
    "usable_worker_capacity": usable_worker_capacity,
    "planned_replicas": replicas,
}
capacity_plan


#### 4.2.3．负载—容量证据图

**学习问题。** 在本章已有的请求到达率、平均输入/输出长度、单 Worker 实测吞吐和目标利用率记录下，最少多少个 Ready 副本才能使目标可用容量覆盖平均 Token 需求？相邻整数副本方案分别留下多少容量余量或缺口？

本图直接消费 `capacity_plan`，不添加流量采样点。横轴上的每个位置是一个整数副本决策，不是观测时间；左图比较相同需求与各副本数对应的目标可用容量，右图显示二者之差。当前记录给出 $D=18\times(620+180)=14{,}400$ tokens/s，单 Worker 目标可用容量为 $3{,}800\times0.70=2{,}660$ tokens/s。

运行前应明确以下形状与数值不变量：

- `replica_candidates`、`usable_capacities` 与 `capacity_headroom` 均为长度 `[planned_replicas + 2]` 的一维序列；当前记录对应长度 8。
- 对任意整数副本数 $r$，目标可用容量严格等于 $rQ_{worker}u$，余量严格等于容量减去同一个 $D_{token}$。
- `planned_replicas` 必须等于第一个满足 `usable_capacity >= token_demand` 的整数点，也必须与 `my_plan_replicas(**capacity_profile)` 一致。
- 当前记录下 5 个副本提供 13,300 tokens/s，仍缺 1,100 tokens/s；6 个副本提供 15,960 tokens/s，余量为 1,560 tokens/s。


In [ ]:
# 只枚举整数副本决策，并由现有容量记录计算需求、容量与余量。
import matplotlib.pyplot as plt

replica_candidates = list(range(1, capacity_plan["planned_replicas"] + 3))
usable_capacities = [
    replica_count * capacity_plan["usable_worker_capacity"]
    for replica_count in replica_candidates
]
capacity_headroom = [
    capacity - capacity_plan["token_demand_per_second"]
    for capacity in usable_capacities
]
first_feasible_replica = next(
    replica_count
    for replica_count, capacity in zip(replica_candidates, usable_capacities)
    if capacity >= capacity_plan["token_demand_per_second"]
)

if not (len(replica_candidates) == len(usable_capacities) == len(capacity_headroom)):
    raise RuntimeError("副本、容量与余量序列的形状不一致")
if first_feasible_replica != capacity_plan["planned_replicas"]:
    raise RuntimeError("图中首个可行副本数与容量规划函数不一致")
for replica_count, capacity, headroom in zip(
    replica_candidates, usable_capacities, capacity_headroom
):
    expected_capacity = replica_count * capacity_plan["usable_worker_capacity"]
    if capacity != expected_capacity:
        raise RuntimeError("目标可用容量未按副本数线性计算")
    if headroom != capacity - capacity_plan["token_demand_per_second"]:
        raise RuntimeError("容量余量未满足 capacity - demand 契约")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
axes[0].plot(
    replica_candidates, usable_capacities, marker="o", linewidth=2.2,
    color="#2563eb", label="目标可用容量",
)
axes[0].axhline(
    capacity_plan["token_demand_per_second"], color="#dc2626",
    linestyle="--", linewidth=2.0, label="当前平均 Token 需求",
)
axes[0].axvline(
    capacity_plan["planned_replicas"], color="#0f766e",
    linestyle=":", linewidth=2.0, label="首个可行整数副本",
)
axes[0].set_xticks(replica_candidates)
axes[0].set_xlabel("Ready 副本数")
axes[0].set_ylabel("tokens/s")
axes[0].set_title("固定负载与整数副本容量")
axes[0].grid(alpha=0.25)
axes[0].legend()

bar_colors = ["#dc2626" if headroom < 0 else "#16a34a" for headroom in capacity_headroom]
axes[1].bar(replica_candidates, capacity_headroom, color=bar_colors, width=0.72)
axes[1].axhline(0.0, color="#0f172a", linewidth=1.2)
axes[1].set_xticks(replica_candidates)
axes[1].set_xlabel("Ready 副本数")
axes[1].set_ylabel("容量余量（tokens/s）")
axes[1].set_title("目标容量减去当前需求")
axes[1].grid(axis="y", alpha=0.25)
for replica_count, headroom in zip(replica_candidates, capacity_headroom):
    axes[1].text(
        replica_count, headroom, f"{headroom:,.0f}",
        ha="center", va="bottom" if headroom >= 0 else "top", fontsize=8,
    )
fig.suptitle("由本章容量记录计算的负载—容量证据")
plt.show()

print(
    "容量图契约：",
    {
        "token_demand_per_second": capacity_plan["token_demand_per_second"],
        "usable_worker_capacity": capacity_plan["usable_worker_capacity"],
        "first_feasible_replica": first_feasible_replica,
        "planned_headroom": capacity_headroom[
            capacity_plan["planned_replicas"] - 1
        ],
    },
)


**应观察到的结论。** 蓝色容量线随整数副本数线性增长，前 5 个决策点仍低于固定需求线；第 6 个点首次越过需求，并在余量图中由负值转为正值。这同时解释了向上取整的必要性，以及目标利用率如何把单 Worker 原始吞吐折减为可规划容量。

**不可误读的边界。** 该图是由一条容量记录和基础公式得到的预算证据，不是实测队列、TTFT、TPOT 或端到端延迟曲线，也不证明 6 个副本已经满足尾延迟 SLO。当前输入仍是本章用于说明方法的假设压测记录；迁移到生产时需要替换为目标制品、引擎、硬件、长度分桶和真实流量窗口的测量值，并用排队与尾延迟观测校准目标利用率。图中也未表达 Prefill/Decode 差异、动态批处理效率、冷启动、故障冗余或突发流量，这些因素可能要求更多 Ready 容量。


#### 4.2.4．扩缩容信号的优先级

| 信号 | 用途 | 注意点 |
|---|---|---|
| 最老请求排队时间 | 直接反映等待体验 | 适合作为快速扩容和拒绝新流量的信号 |
| 队列中的预计 Token | 比请求数更接近真实工作量 | 需要稳定的 Token 估算口径 |
| 每副本在途序列数 | 表示调度槽位压力 | 必须排除尚未 Ready 的副本 |
| TTFT、TPOT 尾分位 | 验证用户体验是否退化 | 是滞后结果，不宜单独驱动扩容 |
| 设备与显存利用率 | 定位资源瓶颈和异常 | 高利用率可能健康，低利用率也可能在等待数据或通信 |
| Ready 副本与启动耗时 | 判断真实可用容量与提前量 | 期望副本不等于可服务副本 |

控制器可以同时计算多个信号对应的期望副本，并采用最保守的结果。缩容前先进入 Draining；若模型常驻成本高，可维护按流量时段预热的容量池。


### 4.3．指标、日志与追踪的关联

![架构图：遥测信号经采集、存储和诊断形成扩缩容与发布反馈](assets/figures/60_inference_deployment/observability-loop.svg)

[TikZ 源文件](assets/figures/60_inference_deployment/observability-loop.tex)

三类信号各有职责：指标回答‘是否异常、影响多大’，追踪回答‘时间花在哪一跳’，日志回答‘发生了什么状态变化’。它们通过 `request_id`、`trace_id`、制品修订和服务实例关联。


#### 4.3.1．LLM 服务的核心时间指标

设请求被 Gateway 接纳的时间为 $t_{admit}$，开始执行为 $t_{start}$，首个 Token 到达客户端为 $t_{first}$，最后一个 Token 到达为 $t_{last}$，输出 Token 数为 $N_{out}$：

$$
T_{queue} = t_{start} - t_{admit}
$$

$$
TTFT = t_{first} - t_{admit}
$$

当 $N_{out} > 1$ 时：

$$
TPOT = \frac{t_{last} - t_{first}}{N_{out} - 1}
$$

端到端延迟为 $t_{last}-t_{admit}$。TTFT 包含排队与首轮计算，TPOT 描述首 Token 之后的平均生成间隔；必须同时观察 P50、P95、P99 和请求长度分桶。P50 表示典型体验，P95/P99 用于发现少数慢请求与容量风险；分位数本身不是固定门槛，门槛来自产品 SLO。P99 在小样本或过短窗口中非常不稳定，报告必须带请求数、时间窗口和长度/租户切片，低流量服务可使用更长窗口或直接报告最慢样本。


#### 4.3.2．遥测最小集合

| 信号 | 推荐字段 | 不应记录的内容 |
|---|---|---|
| 指标 | 请求数、错误码、取消、排队、TTFT、TPOT、Token 吞吐、Ready 副本、加载耗时、显存水位 | `request_id`、用户标识、原始路径参数等高基数标签 |
| 日志 | 时间、级别、服务、修订、区域、请求状态、Token 计数、终止原因、`trace_id` | 默认不记录 Prompt、完整输出、密钥和个人数据 |
| 追踪 | Gateway、Router、Queue、Scheduler、Worker、Streaming Span | 不为每个 Token 创建 Span；用事件或聚合属性控制体量 |

标签基数直接决定时序数量和成本。模型修订、区域、状态码等有限集合适合作为标签；请求、会话、用户和 Prompt 摘要放进受控日志或追踪，并设置采样、脱敏和保留策略。

告警以用户影响为主线：通过错误预算消耗、队列与尾延迟识别影响，再利用副本状态、加载失败、资源水位和组件追踪定位原因。


### 4.4．故障域、背压与恢复验证

![架构图：两个独立故障域通过有界队列和入口背压隔离容量](assets/figures/60_inference_deployment/failure-domains.svg)

[TikZ 源文件](assets/figures/60_inference_deployment/failure-domains.tex)

区域、可用区、模型修订、租户等级和 Worker 池都是故障隔离边界。共享一个无限队列会把局部慢实例扩散成全局超时；共享同一份可变配置会让一次错误更新同时破坏所有副本。


#### 4.4.1．分层背压

1. **入口预算**：限制请求体、上下文长度、并发、每分钟请求与 Token；
2. **接纳控制**：根据预计 Token、截止时间和 Ready 容量，决定接收、降级或拒绝；
3. **有界队列**：每个模型与租户有明确上限，使用公平调度防止大请求长期占用；
4. **执行预算**：Worker 接近显存或序列槽位上限时，Scheduler 停止继续分配；
5. **流式背压**：限制连接发送缓冲和慢消费者存活时间；
6. **负载卸载**：在尾延迟失控前返回稳定错误，保护已经接纳的请求。

降级必须保持语义透明，例如切换到明确标识的较小模型、降低最大输出长度或关闭非必要候选采样。不能静默改变模型修订或安全策略。

这些预算没有通用常数：最大并发/序列槽位由目标模型的 KV Cache、工作区、显存水位和延迟曲线压测确定；队列更适合用预计 Token 而非请求数封顶，并保证最老请求仍可能在 Deadline 内完成；RPM/TPM 和租户并发由配额与公平策略确定；流式缓冲和慢消费者超时由网络速率、内存预算与客户端行为确定。扩缩容的最小热副本、观测窗口和冷却时间则要覆盖模型启动时间与流量波动。调大这些值可提高瞬时接纳量，却会增加排队、尾延迟和故障影响面；调小会更早拒绝请求但保护已接纳流量。


#### 4.4.2．重试、超时和取消矩阵

| 场景 | 服务端动作 | 客户端动作 |
|---|---|---|
| 入队前暂时不可用 | 可在同一故障域外重选一次容量 | 使用抖动退避并服从总期限 |
| 已入队但尚未执行 | 期限到期后移出队列并终止计费 | 根据错误码决定是否重新发起 |
| 已开始执行但尚未流式输出 | 仅在请求幂等且剩余期限足够时受控重试 | 保持相同幂等键 |
| 已输出 Token | 不自动重试，不迁移到另一修订 | 继续消费、主动取消或向用户报告中断 |
| 客户端断开 | 传播取消并释放队列、计算和缓冲 | 重新连接视为新请求 |
| Worker 失联 | 标记实例 NotReady，终止其在途请求并记录影响 | 根据是否已收到 Token 决定重试 |

总期限比每跳超时更重要。每一跳都应知道剩余预算，避免 Gateway 已经放弃请求，而 Worker 仍在继续生成。表中的“域外重选一次”是限制重试放大的保守示例，不是通用推荐次数：最大 Attempts、Connect/Read/Queue/Model Timeout、退避基数与上限、Jitter 都要由幂等性、失败相关性和剩余 Deadline 决定，并由同一个总期限封顶。重试越多越可能掩盖瞬时故障，也越会在过载时放大流量；已经输出 Token 的请求自动重试次数必须为零。


#### 4.4.3．故障演练与状态转换

| 演练 | 预期证据 |
|---|---|
| 权重拉取变慢或摘要不匹配 | Startup 不通过；实例不进入流量池；发布停止 |
| 单个 Worker 卡死 | Liveness 触发重建；故障域内其余请求继续；影响请求可追踪 |
| 队列突然增长 | 接纳控制生效；扩容启动；已接纳请求尾延迟受控 |
| 指标后端不可用 | 推理路径继续；本地缓冲有界；遥测失败不会耗尽内存 |
| 候选修订质量回归 | 自动或人工门禁停止放量；路由恢复稳定修订 |
| 计划缩容 | Worker 先退出 Ready，再排空并在宽限期内终止 |
| 故障域整体失效 | 全局入口按剩余容量限流；不会把全部请求瞬间压向健康域 |

演练不仅看服务是否恢复，还要核对错误预算消耗、检测时间、路由收敛、取消成功率、证据完整性和恢复时间目标。


## 5．迁移到生产库

### 5.1．原理对象与生产组件映射

系统类章节通常没有单一库能够覆盖完整链路。迁移过程保持制品、请求、状态和容量语义不变，再由生产组件分别承接：

| 原理对象 | 生产对象 | 需要固定的契约 |
|---|---|---|
| 不可变制品 Manifest | 模型注册表、对象存储、签名与物料清单 | 内容摘要、签名、基座与派生关系、运行时镜像 |
| Pydantic 请求模型 | OpenAPI/JSON Schema、gRPC Protobuf、服务 SDK | 字段语义、错误码、期限、幂等与流式终止事件 |
| 请求状态与路由快照 | Gateway、Router、Scheduler 与模型服务器 | 活动部署解析、取消传播、队列边界和健康状态 |
| 容量公式 | 压测套件、Autoscaler 与编排平台 | 长度分布、目标硬件、TTFT/TPOT、启动提前量 |
| 关联标识 | OpenTelemetry 指标、日志与追踪 | `request_id`、`trace_id`、制品修订和采样策略 |

### 5.2．迁移验证

迁移后的组件使用同一组制品清单、请求样例和状态转换测试。Schema 兼容性、取消与期限传播、固定修订路由、容量估算误差和遥测关联构成最小等价性证据。

#### 5.2.1．TP 兼容性作为制品—运行时契约

部署控制面不能仅以 GPU 数量判断 Tensor Parallel（TP，张量并行）是否可用。每个候选修订在进入 Worker 预热前，应读取模型 Config、目标运行时版本与 TP 大小，并完成以下预检：

1. 常规 MHA/GQA/MQA 模型的 `hidden_size` 能被 `num_attention_heads` 整除，Query Head 能按 TP 均匀分片；
2. GQA/MQA 的 KV Head 是均匀分片还是由运行时复制，二者对应不同的 KV Cache 峰值；
3. Hidden/FFN/Vocabulary、量化 Group Size、MoE Expert、Checkpoint 与 Kernel 布局满足锁定运行时的约束；
4. GPU 总数能组成完整 TP Group，并为 Replica、PP、故障域和维护容量保留合法拓扑。

预检失败属于不可重试的配置错误，不应让实例进入 Readiness 重试或接收流量。通过预检也只证明形状和布局可加载；峰值显存、通信占比、TTFT、TPOT、吞吐、质量与故障恢复仍须实测。Head 分片、KV 复制及非整除场景的理论与容量模型见 [A50_inference_optimization.ipynb](A50_inference_optimization.ipynb) 5.9 节。


## 6．生产边界

### 6.1．生产验收清单

#### 6.1.1．制品与可重现性

- 权重、配置、Tokenizer、模板、运行时和报告都有内容摘要；
- 线上响应能定位实际制品修订；
- 不使用浮动标签，旧修订仍可拉取并具备回滚容量。

#### 6.1.2．接口与生命周期

- 非流式、流式、终止事件、取消、期限、幂等和错误码语义固定；
- Startup、Liveness、Readiness 各自回答不同问题；
- 下线先停止新流量，再排空，最后终止。

#### 6.1.3．容量与可靠性

- 基准覆盖真实长度分布、并发、目标硬件和制品修订；
- TP 候选已通过模型 Config、Head/KV Head、层维度、量化格式、Kernel 与设备拓扑的锁定版本预检；若复制 KV Head，容量模型使用实际每卡 KV 比例而不是默认 `1 / TP`；
- 扩缩容同时观察 Token 负载、队列、Ready 容量、TTFT 和 TPOT；
- 队列有界，租户公平，过载会显式拒绝而不是无限等待；
- 跨故障域容量、恢复时间和故障演练达到目标。

#### 6.1.4．最低安全发布门禁

- 制品内容摘要、来源和审批状态均可验证；第三方开放权重制品还要校验 Model Audit Manifest 摘要，并确认其与 Evaluation Manifest 中的 SUT 引用相同；
- Gateway 强制身份认证、租户隔离、最小权限、限流、输入上限和资源配额；
- 工具调用经过独立授权与参数白名单，模型不能直接获得凭证；完整安全验收前，外部写操作、资金操作和权限变更保持关闭；
- Prompt、输出、日志与 Trace 执行敏感数据最小化和脱敏；
- 高严重度安全回归采用独立硬门禁，未通过时默认拒绝发布；审计、能力级熔断、凭证吊销和完整回滚路径已经演练。

#### 6.1.5．变更与运维

- 候选修订经过影子或灰度门禁，质量与系统指标共同决定是否在受控部署范围内晋级；
- 回滚恢复完整契约，不在生成中途切换修订；
- 指标、日志和追踪可关联，且控制标签基数和敏感数据；
- 完整安全保障、残余风险接受和最终放量按 [70_model_safety.ipynb](70_model_safety.ipynb) 验收。

通过本章验收仅表示服务可以进入隔离环境、无副作用影子流量或受限灰度，不代表获准全面生产开放。系统级安全保障与残余风险接受由 [70_model_safety.ipynb](70_model_safety.ipynb) 完成。

容量、SLO 与发布门槛不能从本章示例数字直接复制。应在固定制品、引擎、硬件和真实长度/并发分布下预先确定样本量与观测窗口，报告请求数、分位数置信范围或误差，并把候选版本与稳定基线做同口径比较；低频错误和 P99 需要足够长的窗口。质量或安全高严重度失效使用独立硬门禁，不能被平均延迟或平均成功率抵消。


### 6.2．参考资料

- [Kubernetes：Liveness、Readiness 与 Startup Probes](https://kubernetes.io/docs/concepts/workloads/pods/probes/)
- [Kubernetes：Pod 生命周期与优雅终止](https://kubernetes.io/docs/concepts/workloads/pods/pod-lifecycle/)
- [Kubernetes：Horizontal Pod Autoscaling](https://kubernetes.io/docs/concepts/workloads/autoscaling/horizontal-pod-autoscale/)
- [OpenTelemetry：Signals](https://opentelemetry.io/docs/concepts/signals/)
- [OpenTelemetry Collector：Architecture](https://opentelemetry.io/docs/collector/architecture/)
- [Prometheus：Instrumentation](https://prometheus.io/docs/practices/instrumentation/)
- [OpenAPI Specification](https://spec.openapis.org/oas/latest.html)
- [WHATWG：Server-Sent Events](https://html.spec.whatwg.org/multipage/server-sent-events.html)
- [RFC 9110：HTTP Semantics](https://www.rfc-editor.org/rfc/rfc9110)

生产使用前，以部署环境中锁定版本对应的官方文档和组织内部 SLO 为准。


### 6.3．部署闭环

模型部署可以归纳为四个相互约束的闭环：

1. **制品闭环**：清单把权重、Tokenizer、模板、运行时和报告绑定为不可变修订；
2. **请求闭环**：Gateway、Router、Scheduler 和 Worker 共享请求身份、期限、流式状态与取消语义；
3. **容量闭环**：Token 负载和队列预测需求，TTFT、TPOT 与错误预算校验体验，Ready 容量决定真实供给；
4. **变更闭环**：影子、灰度和滚动逐步暴露风险，完整契约回滚恢复稳定状态。

可观测性为四个闭环提供证据，故障域与背压阻止局部问题演化为级联失败，模型安全则约束每一个入口、制品和变更。推理优化决定单个 Worker 的执行性能与资源效率；部署系统决定这些能力能否稳定、可控地服务真实流量。
